# Plot aligned trial figures

This notebook aligns one packed session to visual stimulus onset and then builds trial heatmaps, average neural responses, and pupil traces step by step.

## 1. Setup

The alignment helper returns `neu_seq = (n_trials, n_neurons, n_aligned_times)` and `neu_time = (n_aligned_times,)`.

In [ ]:
from pathlib import Path
import os

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)

H5_PATH = repo / 'results_pack' / 'YH21SC' / 'V1_random_01' / 'V1_random_01.h5'
print(H5_PATH)
print('exists:', H5_PATH.exists())

## 2. Load the packed session

Only `/neural_trials` is needed for stimulus-aligned figures.

In [ ]:
import h5py
import numpy as np

with h5py.File(H5_PATH, 'r') as f:
    neural_trials = {key: f['neural_trials'][key][:] for key in f['neural_trials'].keys()}

for key in ['dff', 'time', 'stim_labels', 'camera_time', 'camera_pupil']:
    print(key, neural_trials[key].shape)

## 3. Align trials to stimulus onset

`n_stim=5` keeps five neighboring stimuli on each side in `stim_seq`. `expected='none'` aligns exactly to the current stimulus onset.

In [ ]:
from modules.Alignment import get_stim_response

stim_labels, neu_seq, neu_time, stim_seq, camera_pupil, pre_isi, post_isi = get_stim_response(
    neural_trials,
    expected='none',
    n_stim=5,
)

print('stim_labels:', stim_labels.shape)
print('neu_seq:', neu_seq.shape)
print('neu_time:', neu_time.shape)
print('stim_seq:', stim_seq.shape)
print('camera_pupil:', camera_pupil.shape)
print('pre_isi:', pre_isi.shape, 'post_isi:', post_isi.shape)

## 4. Plot one neuron's trial heatmap

Each row is one trial and each column is time from stimulus onset. Sorting by pre-interval makes interval structure easier to see.

In [ ]:
import matplotlib.pyplot as plt

ROI_ID = 0
order = np.argsort(pre_isi.reshape(-1))
heatmap = neu_seq[order, ROI_ID, :]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), constrained_layout=True)
im = ax.imshow(
    heatmap,
    aspect='auto',
    extent=[neu_time[0], neu_time[-1], heatmap.shape[0], 0],
    cmap='viridis',
)
ax.axvline(0, color='white', lw=1, ls='--')
ax.set_title(f'ROI {ROI_ID} trial heatmap')
ax.set_xlabel('time from stimulus onset (ms)')
ax.set_ylabel('trials sorted by pre interval')
fig.colorbar(im, ax=ax, label='dF/F')
plt.show()

## 5. Plot population mean response by stimulus label

`stim_labels[:, 2]` is `img_seq_label`. Here we compare a few labels that are present in the session.

In [ ]:
pop_trial = np.nanmean(neu_seq, axis=1)
img_labels = stim_labels[:, 2].astype(int)
labels_to_plot = [x for x in [-1, 2, 3, 4, 5] if np.any(img_labels == x)]

fig, ax = plt.subplots(1, 1, figsize=(7, 3.5), constrained_layout=True)
for lbl in labels_to_plot:
    idx = img_labels == lbl
    m = np.nanmean(pop_trial[idx], axis=0)
    s = np.nanstd(pop_trial[idx], axis=0) / np.sqrt(np.sum(idx))
    ax.plot(neu_time, m, lw=2, label=f'label {lbl}, n={np.sum(idx)}')
    ax.fill_between(neu_time, m-s, m+s, alpha=0.15, lw=0)

ax.axvline(0, color='k', lw=1, ls='--')
ax.set_title('Population mean response by image label')
ax.set_xlabel('time from stimulus onset (ms)')
ax.set_ylabel('mean dF/F')
ax.legend(frameon=False)
plt.show()

## 6. Plot neuron heatmap of mean response

This collapses trials first, then sorts neurons by response around stimulus onset.

In [ ]:
mean_neuron = np.nanmean(neu_seq, axis=0)
win = (neu_time >= 0) & (neu_time <= 1000)
order_neuron = np.argsort(np.nanmean(mean_neuron[:, win], axis=1))

fig, ax = plt.subplots(1, 1, figsize=(7, 5), constrained_layout=True)
im = ax.imshow(
    mean_neuron[order_neuron],
    aspect='auto',
    extent=[neu_time[0], neu_time[-1], mean_neuron.shape[0], 0],
    cmap='magma',
)
ax.axvline(0, color='white', lw=1, ls='--')
ax.set_title('Neuron heatmap of mean aligned response')
ax.set_xlabel('time from stimulus onset (ms)')
ax.set_ylabel('neurons sorted by 0-1000 ms response')
fig.colorbar(im, ax=ax, label='mean dF/F')
plt.show()

## 7. Plot aligned pupil response

If camera data is missing, this panel can be all `NaN`. That is still useful to check before using pupil figures.

In [ ]:
pupil_mean = np.nanmean(camera_pupil, axis=0)
pupil_sem = np.nanstd(camera_pupil, axis=0) / np.sqrt(np.sum(np.isfinite(camera_pupil), axis=0))

fig, ax = plt.subplots(1, 1, figsize=(7, 3), constrained_layout=True)
ax.plot(neu_time, pupil_mean, color='black', lw=2)
ax.fill_between(neu_time, pupil_mean-pupil_sem, pupil_mean+pupil_sem, color='black', alpha=0.2, lw=0)
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_title('Aligned pupil trace')
ax.set_xlabel('time from stimulus onset (ms)')
ax.set_ylabel('pupil')
plt.show()

## 8. Save one aligned figure

Set `SAVE_FIG = True` to write the population response figure into `results/notebook_examples`.

In [ ]:
SAVE_FIG = False

if SAVE_FIG:
    out_dir = repo / 'results' / 'notebook_examples'
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(1, 1, figsize=(7, 3.5), constrained_layout=True)
    for lbl in labels_to_plot:
        idx = img_labels == lbl
        ax.plot(neu_time, np.nanmean(pop_trial[idx], axis=0), lw=2, label=f'label {lbl}')
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_xlabel('time from stimulus onset (ms)')
    ax.set_ylabel('mean dF/F')
    ax.legend(frameon=False)
    fig.savefig(out_dir / 'example_aligned_population.svg')
    print(out_dir / 'example_aligned_population.svg')
else:
    print('Skipped saving.')